# Lab 3B: Agente de Remediação com Controle de Acesso Refinado

## Visão Geral
Este laboratório se baseia no Lab 3A, introduzindo controle de acesso baseado em funções usando autenticação JWT e interceptadores Lambda. Implementaremos um padrão de separação de responsabilidades onde usuários SRE podem planejar remediações, mas apenas aprovadores podem executá-las.

## Objetivos de Aprendizado
- Entender autenticação baseada em JWT com Cognito
- Implementar controle de acesso baseado em funções (RBAC) via interceptadores Lambda
- Implantar e configurar gateways AgentCore com autorização personalizada
- Aplicar separação de responsabilidades em fluxos de trabalho de remediação automatizada
- Monitorar e auditar decisões de acesso

## Personas e Modelo de Acesso

### Função SRE
- **Permissões:** Criar e validar planos de remediação
- **Restrições:** Não pode executar remediações
- **Caso de Uso:** Diagnosticar problemas e propor correções

### Função Aprovador
- **Permissões:** Executar e validar remediações
- **Restrições:** Nenhuma (acesso completo)
- **Caso de Uso:** Revisar planos do SRE e executar correções aprovadas

---

# Passo 1: Explicação do Caso de Uso

## Cenário

Sua organização requer **separação de responsabilidades** para mudanças de infraestrutura:

- **Equipe SRE:** Diagnostica problemas e cria planos de remediação
- **Equipe de Aprovadores:** Revisa e executa mudanças aprovadas

**Problema:** Sem controle de acesso, qualquer pessoa pode executar mudanças de infraestrutura, criando riscos de conformidade e segurança.

**Solução:** Usar autenticação JWT + interceptadores Lambda para aplicar permissões baseadas em funções no nível do gateway.

## Por que JWT + Lambda Interceptor?

**Autenticação JWT:**
- Formato de token padrão da indústria
- Contém identidade do usuário e pertencimento a grupos
- Assinado criptograficamente pelo Cognito
- Não requer consultas ao banco de dados

**Lambda Interceptor:**
- Executa na fase REQUEST do gateway
- Examina claims do JWT antes de rotear para o runtime
- Aplica controle de acesso refinado
- Fornece trilha de auditoria via logs do CloudWatch

**Benefícios:**
- Lógica de autorização centralizada
- Sem alterações no código do agente
- Escalável e serverless
- Fácil de auditar e modificar

## Diagrama do Fluxo de Trabalho

```
┌─────────────────────────────────────────────────────────────────┐
│                    Fluxo SRE (Planejamento)                    │
└─────────────────────────────────────────────────────────────────┘

Usuário SRE → Cognito Auth → JWT (groups: ["sre"])
                                ↓
                          Gateway recebe token
                                ↓
                    Lambda Interceptor verifica:
                    - Extrai cognito:groups = ["sre"]
                    - Ferramenta: generate_remediation_plan
                    - Permissão: ✅ PERMITIDA
                                ↓
                          Runtime executa
                                ↓
                    Plano de remediação criado ✅

┌─────────────────────────────────────────────────────────────────┐
│              Fluxo SRE (Tentativa de Execução)                 │
└─────────────────────────────────────────────────────────────────┘

Usuário SRE → Cognito Auth → JWT (groups: ["sre"])
                                ↓
                          Gateway recebe token
                                ↓
                    Lambda Interceptor verifica:
                    - Extrai cognito:groups = ["sre"]
                    - Ferramenta: execute_remediation_step
                    - Permissão: ❌ NEGADA
                                ↓
                    Requisição bloqueada no gateway ❌

┌─────────────────────────────────────────────────────────────────┐
│                 Fluxo do Aprovador (Execução)                  │
└─────────────────────────────────────────────────────────────────┘

Aprovador → Cognito Auth → JWT (groups: ["approvers"])
                                ↓
                          Gateway recebe token
                                ↓
                    Lambda Interceptor verifica:
                    - Extrai cognito:groups = ["approvers"]
                    - Ferramenta: execute_remediation_step
                    - Permissão: ✅ PERMITIDA
                                ↓
                          Runtime executa
                                ↓
                    Remediação executada ✅
```

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

In [ ]:
# Verify Lab 3A completion
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS

try:
    user_pool_id = get_parameter(PARAMETER_PATHS['cognito']['user_pool_id'])
    print("✅ Lab 3A prerequisites verified")
    print(f"   Cognito User Pool: {user_pool_id}")
    print("   Ready to proceed with Lab 3B")
except Exception as e:
    print("❌ Lab 3A not complete. Please run Lab-03a-remediation-agent.ipynb first.")
    raise

---

# Passo 2: Tokens do Cognito e Claims de Grupo

## Objetivo
Entender o que os tokens do Cognito contêm e como os claims de grupo aplicam o controle de acesso.

## Atividades
- [ ] Obter tokens do Cognito para o usuário SRE
- [ ] Obter tokens do Cognito para o usuário Aprovador
- [ ] Decodificar tokens JWT
- [ ] Exibir claims de grupo de ambos os tokens
- [ ] Comparar diferenças entre os payloads dos tokens

In [ ]:
# Imports
import json
import boto3
from datetime import datetime
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
from lab_helpers.lab_03 import decode_jwt, print_token_claims, compare_tokens

cognito = boto3.client('cognito-idp', region_name=AWS_REGION)

### Token do Usuário SRE

In [ ]:
# Authenticate SRE user
user_client_id = get_parameter(PARAMETER_PATHS['cognito']['user_auth_client_id'])
sre_username = get_parameter(PARAMETER_PATHS['cognito']['test_user_email'])
sre_password = get_parameter(PARAMETER_PATHS['cognito']['test_user_password'])

sre_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': sre_username, 'PASSWORD': sre_password}
)

sre_token = sre_response['AuthenticationResult']['AccessToken']
print(f"✅ SRE authenticated: {sre_username}")
print(f"   Token (first 50 chars): {sre_token}")

In [ ]:
# Decode and display SRE token
sre_claims = decode_jwt(sre_token)
print_token_claims(sre_claims, "SRE Token Claims")

### Token do Usuário Aprovador

In [ ]:
# Authenticate Approver user
approver_username = get_parameter(PARAMETER_PATHS['cognito']['approver_user_email'])
approver_password = get_parameter(PARAMETER_PATHS['cognito']['approver_user_password'])

approver_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': approver_username, 'PASSWORD': approver_password}
)

approver_token = approver_response['AuthenticationResult']['AccessToken']
print(f"✅ Approver authenticated: {approver_username}")
print(f"   Token (first 50 chars): {approver_token[:50]}...")

In [ ]:
# Decode and display Approver token
approver_claims = decode_jwt(approver_token)
print_token_claims(approver_claims, "Approver Token Claims")

### Comparar Payloads dos Tokens

In [ ]:
# Compare tokens side-by-side
compare_tokens(sre_claims, approver_claims)

### Conceitos-Chave

**Estrutura do JWT:**
- `header.payload.signature`
- Payload contém claims (atributos do usuário)
- Assinatura valida a autenticidade do token

**Claims de Grupo no Cognito:**
- `cognito:groups` é incluído automaticamente nos tokens
- Lista todos os grupos aos quais o usuário pertence
- Não são necessários atributos personalizados

**Fluxo de Autorização:**
```
Usuário → Cognito → JWT com cognito:groups
                        ↓
                  Gateway recebe token
                        ↓
                Lambda Interceptor extrai cognito:groups
                        ↓
                Mapeia grupos para ferramentas permitidas
                        ↓
                Decisão de Permitir/Negar
```

**Mapeamento de Permissões (a ser implementado no Lambda):**
```python
GROUP_PERMISSIONS = {
    "sre": ["generate_remediation_plan"],
    "approvers": ["execute_remediation_step", "validate_remediation_environment"]
}
```

---

# Passo 3: Implantar o Lambda Interceptor

## Objetivo
Criar e implantar a função Lambda que aplicará o controle de acesso no nível do gateway.

## Lógica do Interceptor
O Lambda examina as requisições recebidas e:
- [ ] Extrai o JWT dos headers da requisição
- [ ] Valida a assinatura do JWT
- [ ] Verifica os claims de grupo
- [ ] Aplica controle de acesso baseado em ações
- [ ] Retorna a decisão de permitir/negar

### Revisar o Código do Interceptor

In [ ]:
# Review interceptor Lambda code
with open('lab_helpers/lab_03/interceptor-request.py', 'r') as f:
    print(f.read())

### Implantar a Função Lambda

In [ ]:
# Deploy interceptor Lambda in us-west-2
from lab_helpers.lab_03 import deploy_interceptor
from lab_helpers.parameter_store import put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION, WORKSHOP_NAME

print("📦 Deploying interceptor Lambda...")
function_arn = deploy_interceptor(region=AWS_REGION, prefix=WORKSHOP_NAME)

# Store for later use
put_parameter(PARAMETER_PATHS['lab_03b']['interceptor_function_arn'], function_arn)

print(f"\n✅ Interceptor ready: {function_arn}")


**Observação:** Testaremos o interceptor nos Passos 8-10 invocando o gateway com tokens de diferentes usuários.

---

# Passo 4: Limpar o Gateway do Lab 3A

## Objetivo
Excluir o gateway do Lab 3A porque ele não possui a configuração do Lambda Interceptor necessária para o controle de acesso refinado.

## Por que Excluir o Gateway?

**Configuração do Gateway do Lab 3A:**
- Sem autenticação necessária
- Sem interceptor anexado
- Acesso aberto a todas as ferramentas

**Requisitos do Gateway do Lab 3B:**
- Autenticação JWT via Cognito
- Lambda Interceptor na fase REQUEST
- Aplicação de controle de acesso baseado em funções

**Por que Não Atualizar?**
A configuração do interceptor do gateway não pode ser modificada após a criação—ela deve ser definida durante a implantação inicial. Portanto, precisamos:
1. Excluir o gateway do Lab 3A
2. Criar um novo gateway com configuração de interceptor
3. Reutilizar o runtime existente (sem alterações necessárias)

**O que Será Excluído:**
- ✅ Gateway (requer configuração de interceptor)
- ✅ Targets do gateway (excluídos automaticamente com o gateway)

**O que Será Reutilizado:**
- ✅ Runtime (mesma lógica do agente, gateway diferente)
- ✅ Cognito User Pool (já configurado)
- ✅ Lambda Interceptor (implantado no Passo 3)

### Excluir o Gateway do Lab 3A

In [ ]:
import json
import boto3
import time

# Initialize bedrock-agentcore-control client
agentcore_client = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)

# List all gateways
print("🔍 Listing all gateways...")
response = agentcore_client.list_gateways()
gateways = response.get('items', [])

if not gateways:
    print("ℹ️  No gateways found")
else:
    print(f"📋 Found {len(gateways)} gateway(s)\n")
    
    # Filter to only Lab 3A gateways (to be replaced by Lab 3B)
    lab_3a_gateways = [gw for gw in gateways if 'aiml301-remediation-gateway' in gw.get('name', '')]
    
    if not lab_3a_gateways:
        print("ℹ️  No Lab 3A gateways to delete")
    else:
        print(f"🗑️  Deleting {len(lab_3a_gateways)} Lab 3A gateway(s)\n")
        
        # Delete each Lab 3A gateway
        for gateway in lab_3a_gateways:
            gateway_id = gateway['gatewayId']
            gateway_name = gateway.get('name', 'N/A')
            
            print(f"🗑️  Deleting: {gateway_name} ({gateway_id})")
            
            try:
                # First, list and delete all targets
                targets_response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
                targets = targets_response.get('items', [])
                
                for target in targets:
                    target_id = target['targetId']
                    print(f"   🎯 Deleting target: {target_id}")
                    agentcore_client.delete_gateway_target(
                        gatewayIdentifier=gateway_id,
                        targetId=target_id
                    )
                
                # Wait for all targets to be deleted
                if targets:
                    print(f"   ⏳ Waiting for targets to be deleted...")
                    for _ in range(30):
                        time.sleep(2)
                        check = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)
                        if len(check.get('items', [])) == 0:
                            break
                
                # Now delete the gateway
                agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
                print(f"   ✅ Deleted")
                
            except Exception as e:
                print(f"   ❌ Error: {e}")
        
        print(f"\n✅ Cleanup complete")


**Observação:** O runtime é reutilizado do Lab 3A—não é necessário excluí-lo.

---

# Passo 5: Criar Novo Gateway e Target com Autenticação JWT

## Objetivo
Implantar um novo gateway configurado com autenticação JWT e Lambda Interceptor.

### Configuração do Gateway

**Detalhes do Gateway:**
- Nome: `interceptor-gateway-jwt-[random]`
- Protocolo: MCP
- Tipo de Autorizador: CUSTOM_JWT
- Pontos de Interceptação: REQUEST

**Configuração JWT:**
```json
{
  "authorizerType": "CUSTOM_JWT",
  "discoveryUrl": "https://cognito-idp.us-west-2.amazonaws.com/us-west-2_POOL_ID/.well-known/openid-configuration",
  "allowedClients": [
    "CLIENT_ID_1",
    "CLIENT_ID_2"
  ]
}
```

**Configuração do Interceptor:**
```json
{
  "interceptionPoints": ["REQUEST"],
  "interceptor": {
    "lambda": {
      "arn": "arn:aws:lambda:us-west-2:ACCOUNT:function:custom-interceptor-request"
    }
  },
  "inputConfiguration": {
    "passRequestHeaders": true
  }
}
```


### Criar Gateway

In [ ]:
import random
import string
import time
import boto3
from lab_helpers.parameter_store import get_parameter, put_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
from lab_helpers.lab_03.gateway_setup import AgentCoreGatewaySetup

# Initialize bedrock-agentcore-control client
agentcore_client = boto3.client('bedrock-agentcore-control', region_name= AWS_REGION)

# Get configuration values
user_pool_id = get_parameter(PARAMETER_PATHS['cognito']['user_pool_id'])
user_client_id = get_parameter(PARAMETER_PATHS['cognito']['user_auth_client_id'])
m2m_client_id = get_parameter(PARAMETER_PATHS['cognito']['m2m_client_id'])
interceptor_arn = get_parameter(PARAMETER_PATHS['lab_03b']['interceptor_function_arn'])

# Create Gateway IAM role in us-west-2
gateway_setup = AgentCoreGatewaySetup(region=AWS_REGION, prefix=WORKSHOP_NAME, verbose=False)
role_info = gateway_setup.create_gateway_service_role()
role_arn = role_info['role_arn']
print(f"✅ Gateway role: {role_arn}")

# Generate unique gateway name
suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=10))
gateway_name = f"aiml301-interceptor-gateway-jwt-{suffix}"

print(f"📤 Creating gateway: {gateway_name}")
print(f"   Gateway region: us-west-2")
print(f"   Cognito region: {AWS_REGION}")
print(f"   Role: {role_arn}")
print(f"   Interceptor: {interceptor_arn}")

# Create gateway
response = agentcore_client.create_gateway(
    name=gateway_name,
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": ["2025-03-26"]
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration",
            "allowedClients": [user_client_id, m2m_client_id]
        }
    },
    interceptorConfigurations=[
        {
            "interceptionPoints": ["REQUEST"],
            "interceptor": {
                "lambda": {"arn": interceptor_arn}
            },
            "inputConfiguration": {
                "passRequestHeaders": True
            }
        }
    ],
    roleArn=role_arn
)

gateway_id = response['gatewayId']
gateway_arn = response['gatewayArn']

print(f"\n✅ Gateway created: {gateway_id}")
print(f"   ARN: {gateway_arn}")


In [ ]:
# Wait for gateway to be READY
print("⏳ Waiting for gateway to be ready...")
while True:
    response = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    
    status = response['status']
    if status == 'READY':
        gateway_url = response['gatewayUrl']
        print(f"✅ Gateway ready: {status}")
        print(f"   URL: {gateway_url}")
        break
    elif status == 'FAILED':
        print(f"❌ Gateway creation failed")
        break
    print(f"   Status: {status}")
    time.sleep(5)

# Store for later use
put_parameter(PARAMETER_PATHS['lab_03b']['gateway_id'], gateway_id)
put_parameter(PARAMETER_PATHS['lab_03b']['gateway_url'], gateway_url)


### Inspecionar Configuração do Gateway

In [ ]:
# Display full gateway configuration
result = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)

print("📋 Gateway Configuration:")
print("=" * 70)
print(json.dumps(result, indent=2, default=str))
print("=" * 70)

# Highlight key configuration elements
print("\n🔑 Key Configuration Elements:")
print(f"   Gateway ID: {result['gatewayId']}")
print(f"   Gateway URL: {result['gatewayUrl']}")
print(f"   Protocol: {result['protocolType']}")
print(f"   Authorizer: {result['authorizerType']}")

# JWT Configuration
jwt_config = result['authorizerConfiguration']['customJWTAuthorizer']
print(f"\n🔐 JWT Configuration:")
print(f"   Discovery URL: {jwt_config['discoveryUrl']}")
print(f"   Allowed Clients: {len(jwt_config['allowedClients'])} clients")
for i, client in enumerate(jwt_config['allowedClients'], 1):
    print(f"      {i}. {client}")

# Interceptor Configuration
interceptor_config = result['interceptorConfigurations'][0]
print(f"\n🛡️  Interceptor Configuration:")
print(f"   Lambda ARN: {interceptor_config['interceptor']['lambda']['arn']}")
print(f"   Interception Points: {interceptor_config['interceptionPoints']}")
print(f"   Pass Request Headers: {interceptor_config['inputConfiguration']['passRequestHeaders']}")
print(f"\n   ℹ️  The interceptor will:")
print(f"      • Examine JWT tokens in Authorization header")
print(f"      • Extract cognito:groups claim")
print(f"      • Enforce group-based permissions")
print(f"      • Allow/deny requests before reaching runtime")


Adicionar permissões ao Lambda para que possa ser invocado pelo Interceptor Gateway

In [ ]:
# Update Lambda permission with gateway ARN
import boto3

lambda_client = boto3.client('lambda', region_name=AWS_REGION)
function_name = f"{WORKSHOP_NAME}-interceptor-request"
gateway_arn = result['gatewayArn']

print(f"🔧 Updating Lambda permission with gateway ARN...")

# Remove old permission
try:
    lambda_client.remove_permission(
        FunctionName=function_name,
        StatementId='AllowGatewayInvoke'
    )
    print(f"   Removed old permission")
except:
    pass

# Add new permission with source ARN
lambda_client.add_permission(
    FunctionName=function_name,
    StatementId='AllowGatewayInvoke',
    Action='lambda:InvokeFunction',
    Principal='bedrock-agentcore.amazonaws.com',
    SourceArn=gateway_arn
)

print(f"✅ Lambda permission updated with source ARN")
print(f"   Gateway ARN: {gateway_arn}")

### Configuração do Target

**Detalhes do Target:**
- Nome: `mcp-remediation-target`
- Tipo: MCP Server
- Endpoint: [Endpoint do runtime do passo 6]

**Provedor de Credenciais:**
- Tipo: OAUTH
- Provedor: Cognito MCP Provider

### Criar Target

In [ ]:
# Get runtime endpoint from Lab 3A
runtime_arn = get_parameter(PARAMETER_PATHS['lab_03']['runtime_arn'])
runtime_endpoint = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/runtimes/{runtime_arn.replace(':', '%3A').replace('/', '%2F')}/invocations?qualifier=DEFAULT"
print (f'runtime endpoint: {runtime_endpoint}')
# Get OAuth provider ARN
oauth_provider_arn = get_parameter(PARAMETER_PATHS['lab_03']['oauth2_provider_arn'])

# Create target
response = agentcore_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="mcp-remediation-target",
    description="MCP server target for remediation agent with JWT auth",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": runtime_endpoint
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": oauth_provider_arn,
                    "scopes": []
                }
            }
        }
    ]
)

target_id = response['targetId']
print(f"✅ Target Creating: {target_id}")
print(f"   Runtime: {runtime_arn}")


In [ ]:
# Wait for target to be READY
print("⏳ Waiting for target to be ready...")

target_info = agentcore_client.get_gateway_target(
          gatewayIdentifier=gateway_id,
          targetId=target_id
      )
time.sleep(5)
status = target_info.get('status', 'UNKNOWN')

if status == 'READY':
    print(f"✅ Target is READY")
    print(f"\n🎉 Gateway and target ready for testing!")
    print(f"   Gateway URL: {gateway_url}")
    print(f"   Target ID: {target_id}")
if status == 'FAILED' or status == 'SYNCHRONIZE_UNSUCCESSFUL':
    print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")

### Inspecionar Configuração do Target

In [ ]:
# Display full target configuration
result = agentcore_client.get_gateway_target(
    gatewayIdentifier=gateway_id,
    targetId=target_id
)

print("📋 Target Configuration:")
print("=" * 70)
print(json.dumps(result, indent=2, default=str))
print("=" * 70)

# Highlight key configuration elements
print("\n🔑 Key Configuration Elements:")
print(f"   Target ID: {result['targetId']}")
print(f"   Target Name: {result['name']}")
print(f"   Status: {result['status']}")

# MCP Server Configuration
mcp_config = result['targetConfiguration']['mcp']['mcpServer']
print(f"\n🔌 MCP Server Configuration:")
print(f"   Endpoint: {mcp_config['endpoint'][:80]}...")
print(f"   ℹ️  Points to Lab 3A runtime (reused)")

# Credential Provider Configuration
cred_config = result['credentialProviderConfigurations'][0]
oauth_config = cred_config['credentialProvider']['oauthCredentialProvider']
print(f"\n🔐 Credential Provider Configuration:")
print(f"   Type: {cred_config['credentialProviderType']}")
print(f"   Provider ARN: {oauth_config['providerArn']}")
print(f"   ℹ️  Uses Cognito OAuth provider for machine-to-machine auth")

# Explain the flow
print(f"\n📊 Request Flow:")
print(f"   1. User sends request with JWT token → Gateway")
print(f"   2. Gateway validates JWT (Cognito OIDC)")
print(f"   3. Lambda interceptor checks permissions")
print(f"   4. If allowed → Gateway forwards to Target")
print(f"   5. Target uses OAuth to authenticate with Runtime")
print(f"   6. Runtime executes agent logic")
print(f"   7. Response flows back through Gateway")


---

# Passo 6: Revisar o MCP Runtime

## Objetivo
Revisar o runtime do agente de remediação que será acessado através do gateway.

### Verificar Configuração do Runtime

In [ ]:
# Verify runtime from Lab 3A is available
from lab_helpers.parameter_store import get_parameter
from lab_helpers.constants import PARAMETER_PATHS
from lab_helpers.config import AWS_REGION
import boto3

runtime_arn = get_parameter(PARAMETER_PATHS['lab_03']['runtime_arn'])
runtime_id = runtime_arn.split('/')[-1]

print(f"✅ Runtime from Lab 3A:")
print(f"   ARN: {runtime_arn}")
print(f"   ID: {runtime_id}")
print(f"   Region: {AWS_REGION}")

# Check runtime status using control plane client
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)
try:
    runtime_info = agentcore_control.get_agent_runtime(agentRuntimeId=runtime_id)
    status = runtime_info['status']
    print(f"\n📊 Runtime Status: {status}")
    
    if status == 'READY':
        print(f"   ✅ Runtime is ready for gateway integration")
    elif status == 'FAILED' or status == 'SYNCHRONIZE_UNSUCCESSFUL':
        print(f"❌ Target in ERROR state: {target_info.get('statusReasons', 'No error message')}")
except Exception as e:
    print(f"   ❌ Error checking runtime: {e}")


---

# Passo 7: Regenerar Tokens do Cognito

## Objetivo
Obter tokens JWT atualizados para ambas as personas que sejam válidos para o novo gateway.

### Token do Usuário SRE

In [ ]:
# Regenerate SRE token (in case Step 2 tokens expired)
sre_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': sre_username, 'PASSWORD': sre_password}
)

sre_token = sre_response['AuthenticationResult']['AccessToken']
sre_claims = decode_jwt(sre_token)

print(f"✅ Fresh SRE token generated")
print(f"   Username: {sre_username}")
print(f"   Groups: {sre_claims.get('cognito:groups', [])}")
print(f"   Token (first 50 chars): {sre_token[:50]}...")

### Token do Usuário Aprovador

In [ ]:
# Regenerate Approver token (in case Step 2 tokens expired)
approver_response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME': approver_username, 'PASSWORD': approver_password}
)

approver_token = approver_response['AuthenticationResult']['AccessToken']
approver_claims = decode_jwt(approver_token)

print(f"✅ Fresh Approver token generated")
print(f"   Username: {approver_username}")
print(f"   Groups: {approver_claims.get('cognito:groups', [])}")
print(f"   Token (first 50 chars): {approver_token[:50]}...")

## Enriquecer Contexto da Memória com Informações de Diagnóstico

Vamos obter informações adicionais da nossa memória curada para enriquecer o contexto com informações de diagnóstico.

In [ ]:
agent_memory_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

memory_id = get_parameter(PARAMETER_PATHS['memory']['memory_id'])
memory_session_id = get_parameter(PARAMETER_PATHS['memory']['default_session_id'])

print(memory_id)
print(memory_session_id)
actor_id='diagnostics_agent'

  

#list events added to agent memory, to confirm successful write
params = {
                "memoryId": memory_id,
                "actorId": actor_id,
                "sessionId": memory_session_id,
                "includePayloads": True
            }
# Get all messages
response = agent_memory_client.list_events(**params)
additional_context=""
for event in response.get("events", []):
    payload = event.get('payload', [])
    for i, item in enumerate(payload):
        if 'conversational' in item:
            text = item['conversational']['content']['text']
            additional_context+=text
additional_context

---

# Passo 8: SRE Simula Criação de Plano

## Objetivo
Mostrar que usuários SRE podem criar planos de remediação com sucesso.

## Cenário
O SRE detecta um problema e cria um plano para remediá-lo.

### Executar Criação do Plano

In [ ]:
# Connect to gateway with SRE token
from lab_helpers.lab_03.mcp_client import MCPClient

gateway_url = get_parameter(PARAMETER_PATHS['lab_03b']['gateway_url'])

print(f"🔗 Connecting to gateway as SRE user...")
print(f"   Gateway: {gateway_url}")
print(f"   User: {sre_username}")
print(f"   Groups: {sre_claims.get('cognito:groups', [])}")

sre_client = MCPClient(gateway_url, sre_token)
sre_client.initialize()

print(f"\n✅ Connected to gateway")

In [ ]:
# List available tools
print(f"🔧 Listing available tools...\n")
tools = sre_client.list_tools()

print(f"📋 Available tools ({len(tools)}):")
for tool in tools:
    print(f"   • {tool['name']}")
    if 'description' in tool:
        print(tool)
        desc = tool['description']
        print(f"     {desc[:80]}..." if len(desc) > 80 else f"     {desc}")

In [ ]:
# Invoke remediation planning tool
start_time = datetime.now()
print(f"\n🎯 Invoking generate_remediation_plan tool...\n")
try:
    result = sre_client.call_tool(
        tool_name='mcp-remediation-target___infrastructure_agent',
        arguments={
            'remediation_query': f"""I need help with infrastructure remediation for our CRM application. We're experiencing: {additional_context} """,
            'action_type': 'only_plan'
        }
    )
except Exception as e:
    print(f"❌ Error: {e}")
analysis_time = (datetime.now() - start_time).total_seconds()
print(f"Analysis Time: {analysis_time:.2f} seconds")
print(f"✅ Tool invocation successful!")
print(f"\n📋 Remediation Plan:")
print(f"{result[:500]}..." if len(result) > 500 else result)

### Verificar Sucesso

In [ ]:
print(f"\n✅ SRE User Successfully:")
print(f"   1. Connected to gateway with JWT token")
print(f"   2. Listed available tools")
print(f"   3. Invoked generate_remediation_plan tool")
print(f"   4. Received remediation plan")
print(f"\n🔒 Interceptor allowed this operation because:")
print(f"   • User is in 'sre' group")
print(f"   • Tool 'generate_remediation_plan' is allowed for SRE users")

### Resultado Esperado
✅ Plano criado com sucesso com o token SRE

---

# Passo 9: SRE Tenta Executar (Acesso Negado)

## Objetivo
Demonstrar que usuários SRE NÃO PODEM executar remediações—o interceptor os bloqueia.

## Cenário
O SRE tenta executar o plano que criou.

### Tentativa de Execução

In [ ]:
# Invoke remediation planning tool
print(f"\n🎯 Invoking execute_remediation_step tool...\n")

result = sre_client.call_tool(
    tool_name='mcp-remediation-target___infrastructure_agent',
    arguments={
            "remediation_query": f"""
            Help me change the read and write capacity to on-demand for the DynamoDb tables in my CRM application, based on this diagnostic information: {additional_context}
            """,
            "action_type": "only_execute"
        }
)

### Analisar Negação de Acesso

In [ ]:
# Check Lambda logs to see the interceptor rejection
# Since it may take few minutes for logs to reach cloudwatch, please wait and retry
import boto3
from datetime import datetime

logs = boto3.client('logs', region_name=AWS_REGION)
log_group = '/aws/lambda/aiml301_sre_agentcore-interceptor-request'

streams = logs.describe_log_streams(
    logGroupName=log_group,
    orderBy='LastEventTime',
    descending=True,
    limit=1
)

print("📋 Interceptor Lambda Logs (Last 20 lines):\n")
stream = streams['logStreams'][0]
events = logs.get_log_events(
    logGroupName=log_group,
    logStreamName=stream['logStreamName'],
    limit=30,
    startFromHead=False
)

for event in events['events'][-20:]:
    msg = event['message'].strip()
    if any(x in msg for x in ['Tool call', 'User groups', 'not authorized', 'Denying']):
        print(msg)

print("\n✅ The interceptor blocked the execute_remediation_step call for SRE user")

### Resultado Esperado
❌ Acesso negado - Rejeição do interceptor visível

---

# Passo 10: Aprovador Executa e Valida

## Objetivo
Mostrar que usuários aprovadores podem executar e validar remediações.

### Parte A: Aprovador Executa a Correção

In [ ]:
approver_client = MCPClient(gateway_url, approver_token)
approver_client.initialize()

In [ ]:
# Invoke remediation planning tool
print(f"\n🎯 Invoking execute_remediation_step tool...\n")
start_time = time.time()
try:
    result = approver_client.call_tool(
        tool_name='mcp-remediation-target___infrastructure_agent',
        arguments={
            "remediation_query": f"""
            This is a test invocation to validate access to infrastructure_agent and tool. Please do a health check and confirm your status.
            """,
            "action_type": "only_execute"
        }
    )
except Exception as e:
    print(f"❌ Error: {e}")
end_time = time.time()
print(f"  ⏰ Total time taken: {end_time - start_time:.2f} seconds")

### Parte B: Aprovador Valida a Correção

### Resultado Esperado
✅ Remediação executada e validada com o token do Aprovador

---

# Limpeza

## Objetivo
Remover os recursos do Lab 3B preservando a infraestrutura base do Lab 3A.

**Recursos Excluídos:**
- Gateway com autenticação JWT
- Função Lambda Interceptor
- Role de execução do Lambda

**Recursos Preservados:**
- AgentCore Runtime (do Lab 3A)
- Cognito User Pool e usuários
- OAuth2 Credential Provider
- Entradas no Parameter Store

In [ ]:
from lab_helpers.lab_03 import cleanup_lab_03b
from lab_helpers.config import AWS_REGION

#cleanup_lab_03b(region_name=AWS_REGION, verbose=True)

---

# Referências
- Lab 3A: Fundamentos do Agente de Remediação
- Configuração JWT do Cognito
- Documentação do AgentCore Gateway
- Padrões de Lambda Interceptor